# 00 — Data Audit

Verify all datasets, schema, date ranges, NaN rates. Establish DuckDB catalog view.

In [1]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from dotenv import load_dotenv
load_dotenv('../.env')

import duckdb
import pandas as pd
import numpy as np

X9 = os.environ.get('X9_MOUNT', '/Volumes/Crucial X9')
CATALOG = os.environ.get('DUCKDB_CATALOG', f'{X9}/data/catalog.duckdb')
SP500_DIR = os.path.join('..', 'datasets', 'sp500_holdings')

db = duckdb.connect(CATALOG, read_only=True)
print('Connected to DuckDB catalog:', CATALOG)


Connected to DuckDB catalog: /Volumes/Crucial X9/data/catalog.duckdb


## 1. OHLCV Universe

In [2]:
from src.data.universe_builder import get_universe_tickers

# Get all SP500 tickers 2010-2024 to limit the DuckDB scan
sp500_tickers = get_universe_tickers(SP500_DIR, start_year=2010, end_year=2024)
print(f'SP500 union tickers (2010-2024): {len(sp500_tickers)}')

ticker_list = "', '".join(sorted(sp500_tickers))
query = f"""
    SELECT
        ticker,
        timestamp::DATE AS date,
        FIRST(open ORDER BY timestamp)  AS open,
        MAX(high)                        AS high,
        MIN(low)                         AS low,
        LAST(close ORDER BY timestamp)   AS close,
        SUM(volume)                      AS volume
    FROM equities_ohlcv_1m
    WHERE ticker IN ('{ticker_list}')
      AND timestamp >= '2010-01-01'
      AND timestamp <  '2025-01-01'
    GROUP BY ticker, timestamp::DATE
    ORDER BY ticker, date
"""
print('Running daily resample query (may take ~1 min)...')
daily = db.execute(query).fetchdf()
print(f'Shape: {daily.shape}')
print(f'Date range: {daily.date.min()} → {daily.date.max()}')
print(f'Unique tickers: {daily.ticker.nunique()}')
print()
print('NaN rates per column:')
print((daily.isna().mean() * 100).round(2).rename('nan_%'))


SP500 union tickers (2010-2024): 674
Running daily resample query (may take ~1 min)...


Shape: (2612171, 7)
Date range: 2010-01-01 00:00:00 → 2024-12-31 00:00:00
Unique tickers: 661

NaN rates per column:
ticker    0.0
date      0.0
open      0.0
high      0.0
low       0.0
close     0.0
volume    0.0
Name: nan_%, dtype: float64


## 2. Factor Data (FF5 + Momentum)

In [3]:
# Load Fama-French 5 factors + Momentum
ff5 = db.execute("SELECT * FROM ff_famafrench_ff5_daily").fetchdf()
ff5['date'] = pd.to_datetime(ff5['date']).dt.normalize()
ff5 = ff5.rename(columns={'Mkt-RF': 'mkt_rf', 'SMB': 'smb', 'HML': 'hml',
                            'RMW': 'rmw', 'CMA': 'cma', 'RF': 'rf'})

mom = db.execute("SELECT * FROM ff_famafrench_mom_daily").fetchdf()
mom['date'] = pd.to_datetime(mom['date']).dt.normalize()
print('Mom columns:', mom.columns.tolist())
mom = mom.rename(columns={c: 'mom' for c in mom.columns if c.lower() not in ['date']})

factors = ff5.merge(mom[['date', 'mom']], on='date', how='left')
factors = factors.set_index('date').sort_index()

# Ken French data is in %, convert to decimal
factors = factors / 100.0

mask = (factors.index >= '2010-01-01') & (factors.index <= '2024-12-31')
factors_2010 = factors.loc[mask]
print(f'FF5+Mom factors: {factors_2010.shape}')
print(f'Date range: {factors_2010.index.min().date()} → {factors_2010.index.max().date()}')
print(f'NaN counts:\n{factors_2010.isna().sum()}')
print(f'\nSample:\n{factors_2010.tail(3)}')


Mom columns: ['Mom', 'date']
FF5+Mom factors: (3774, 7)
Date range: 2010-01-04 → 2024-12-31
NaN counts:
mkt_rf    0
smb       0
hml       0
rmw       0
cma       0
rf        0
mom       0
dtype: int64

Sample:
            mkt_rf     smb     hml     rmw     cma      rf     mom
date                                                              
2024-12-27 -0.0117 -0.0044  0.0057  0.0039  0.0004  0.0002 -0.0085
2024-12-30 -0.0109  0.0025  0.0074  0.0055  0.0014  0.0002  0.0009
2024-12-31 -0.0046  0.0032  0.0072  0.0033  0.0002  0.0002 -0.0107


## 3. SP500 Holdings (Survivorship Bias Check)

In [4]:
from src.data.universe_builder import build_sp500_universe

universe = build_sp500_universe(SP500_DIR, start_date='2010-01-01', end_date='2024-12-31')
print(f'Universe table shape: {universe.shape}')

member_count = universe.groupby(universe['date'].dt.year)['ticker'].nunique()
print('\nS&P 500 member count by year:')
print(member_count.to_string())

# Sanity check: major tickers
for ticker in ['AAPL', 'MSFT', 'JPM', 'XOM']:
    years = universe[universe['ticker'] == ticker]['date'].dt.year.unique()
    print(f'  {ticker}: in index for years {sorted(years)[:5]}...')


Universe table shape: (1968502, 3)

S&P 500 member count by year:
date
2010    433
2011    434
2012    436
2013    438
2014    443
2015    450
2016    458
2017    460
2018    465
2019    475
2020    482
2021    486
2022    492
2023    492
2024    494
  AAPL: in index for years [np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014)]...


  MSFT: in index for years [np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014)]...
  JPM: in index for years [np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014)]...
  XOM: in index for years [np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014)]...


## 4. FRED Macro (VIX, 2s10s, DXY)

In [5]:
fred = db.execute("SELECT * FROM fred_macro").fetchdf()
fred['date'] = pd.to_datetime(fred['date']).dt.normalize()

# Pivot to wide format
macro = fred.pivot_table(index='date', columns='series_id', values='value', aggfunc='last')
macro = macro.rename(columns={
    'VIXCLS':   'vix',
    'T10Y2Y':   'term_spread_2s10s',
    'DTWEXBGS': 'usd_dxy',
})
macro = macro[['vix', 'term_spread_2s10s', 'usd_dxy']].sort_index()

mask_train = (macro.index >= '2010-01-01') & (macro.index <= '2021-12-31')
macro_train = macro.loc[mask_train]

calm_days   = (macro_train['vix'] <= 20).sum()
stress_days = (macro_train['vix'] > 20).sum()
print(f'Macro data shape (2010-2021): {macro_train.shape}')
print(f'VIX calm days (≤20): {calm_days}  ({calm_days/(calm_days+stress_days)*100:.1f}%)')
print(f'VIX stress days (>20): {stress_days}  ({stress_days/(calm_days+stress_days)*100:.1f}%)')
print(f'\nNaN counts:\n{macro_train.isna().sum()}')
print(f'\nSample:\n{macro_train.tail(3)}')


Macro data shape (2010-2021): (4383, 3)
VIX calm days (≤20): 2204  (73.0%)
VIX stress days (>20): 817  (27.0%)

NaN counts:
series_id
vix                  1362
term_spread_2s10s    1380
usd_dxy              1401
dtype: int64

Sample:
series_id     vix  term_spread_2s10s   usd_dxy
date                                          
2021-12-29  16.95               0.80  115.3678
2021-12-30  17.33               0.79  115.2830
2021-12-31  17.22               0.79       NaN


## 5. Polymarket Features

In [6]:
poly_schema = db.execute("DESCRIBE poly_market_features").fetchdf()
print('Polymarket features schema:')
print(poly_schema[['column_name','column_type']].to_string(index=False))

poly_meta = db.execute("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT market_id) AS n_markets,
        epoch_ms(MIN(first_trade_ts)::BIGINT) AS first_trade,
        epoch_ms(MAX(last_trade_ts)::BIGINT)  AS last_trade
    FROM poly_market_features
""").fetchdf()
print(f'\n{poly_meta.to_string(index=False)}')
print('\nNote: used only in Exp 6 (cross-domain robustness) — not in equity pipeline.')


Polymarket features schema:
       column_name column_type
         market_id     VARCHAR
      condition_id     VARCHAR
          event_id     VARCHAR
      total_trades      BIGINT
        buy_trades      DOUBLE
       sell_trades      DOUBLE
  total_usd_volume      DOUBLE
total_token_volume      DOUBLE
         avg_price      DOUBLE
         price_std      DOUBLE
         min_price      DOUBLE
         max_price      DOUBLE
              vwap      DOUBLE
buy_sell_usd_ratio      DOUBLE
    first_trade_ts     UBIGINT
     last_trade_ts     UBIGINT
       active_days      DOUBLE

 n_rows  n_markets             first_trade              last_trade
 615257     615257 1970-01-20 07:37:40.169 1970-01-21 13:02:46.927

Note: used only in Exp 6 (cross-domain robustness) — not in equity pipeline.


## 6. Summary Statistics Table

In [7]:
import os

summary = pd.DataFrame([
    {
        'dataset': 'OHLCV 1-min (SP500 universe)',
        'n_rows': len(daily),
        'n_tickers': daily['ticker'].nunique(),
        'date_min': str(daily['date'].min()),
        'date_max': str(daily['date'].max()),
        'nan_pct': round(daily.isna().mean().mean() * 100, 2),
        'notes': 'resampled to daily; SP500 membership filter applied'
    },
    {
        'dataset': 'FF5+Mom factors',
        'n_rows': len(factors_2010),
        'n_tickers': None,
        'date_min': str(factors_2010.index.min().date()),
        'date_max': str(factors_2010.index.max().date()),
        'nan_pct': round(factors_2010.isna().mean().mean() * 100, 2),
        'notes': 'decimal returns; merged FF5 + Mom'
    },
    {
        'dataset': 'FRED macro (VIX, 2s10s, DXY)',
        'n_rows': len(macro_train),
        'n_tickers': None,
        'date_min': str(macro_train.index.min().date()),
        'date_max': str(macro_train.index.max().date()),
        'nan_pct': round(macro_train.isna().mean().mean() * 100, 2),
        'notes': 'train window only (2010-2021)'
    },
    {
        'dataset': 'SP500 universe membership',
        'n_rows': len(universe),
        'n_tickers': universe['ticker'].nunique(),
        'date_min': str(universe['date'].min().date()),
        'date_max': str(universe['date'].max().date()),
        'nan_pct': 0.0,
        'notes': 'point-in-time annual snapshots'
    },
])

print(summary.to_string(index=False))

out_dir = '../data/processed'
os.makedirs(out_dir, exist_ok=True)
summary.to_csv(f'{out_dir}/audit_summary.csv', index=False)
print(f'\nSaved to {out_dir}/audit_summary.csv')


                     dataset  n_rows  n_tickers            date_min            date_max  nan_pct                                               notes
OHLCV 1-min (SP500 universe) 2612171      661.0 2010-01-01 00:00:00 2024-12-31 00:00:00     0.00 resampled to daily; SP500 membership filter applied
             FF5+Mom factors    3774        NaN          2010-01-04          2024-12-31     0.00                   decimal returns; merged FF5 + Mom
FRED macro (VIX, 2s10s, DXY)    4383        NaN          2010-01-01          2021-12-31    31.51                       train window only (2010-2021)
   SP500 universe membership 1968502      674.0          2010-01-01          2024-12-31     0.00                      point-in-time annual snapshots

Saved to ../data/processed/audit_summary.csv
